<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/Normal_Mode_Analysis_(NMA)_using_ProDy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# Normal Mode Analysis (NMA) using ProDy
# ==========================================================
# Requisitos:
# pip install prody matplotlib numpy seaborn

from prody import *
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

sns.set(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (6,4)

# ==========================================================
# 1️⃣ Upload do arquivo de estrutura (PDB)
# ==========================================================
print("📂 Faça upload do arquivo .pdb da estrutura representativa")
uploaded = files.upload()
pdb_file = [f for f in uploaded if f.endswith('.pdb')][0]

# Carregar estrutura
pdb = parsePDB(pdb_file)
calphas = pdb.select('protein and name CA')

print(f"✅ Estrutura carregada: {pdb_file}")
print(f"✅ {len(calphas)} átomos Cα selecionados para NMA")

# ==========================================================
# 2️⃣ Cálculo dos modos normais (ANM)
# ==========================================================
anm = ANM('Alpha5Beta1_NMA')
anm.buildHessian(calphas)
anm.calcModes(n_modes=20)

# ==========================================================
# 3️⃣ Visualização das frequências e variâncias (painel B)
# ==========================================================
variances = calcSqFlucts(anm)
mode_var = anm.getVariances()
cum_var = np.cumsum(mode_var) / np.sum(mode_var)

plt.figure(figsize=(6,4))
plt.bar(range(1, 21), mode_var[:20]/np.sum(mode_var), color='royalblue')
plt.plot(range(1, 21), cum_var[:20], 'o-', color='royalblue')
plt.axhline(0.8, ls='--', color='k')
plt.axvline(np.argmax(cum_var>0.8)+1, ls='--', color='k')
plt.xlabel('Mode index')
plt.ylabel('Fraction of variance')
plt.title('Fraction of total variance (NMA)')
plt.tight_layout()
plt.show()

# ==========================================================
# 4️⃣ Salvando resultados
# ==========================================================
np.savetxt("NMA_modes.txt", anm.getEigvecs())
np.savetxt("NMA_eigenvalues.txt", anm.getEigvals())

writeNMD('anm_modes.nmd', anm[:10], calphas)  # arquivo NMD para PyMOL/VMD

print("💾 Arquivos gerados:")
print("  • NMA_modes.txt — autovetores dos modos")
print("  • NMA_eigenvalues.txt — autovalores (frequências²)")
print("  • anm_modes.nmd — visualização em PyMOL/VMD")

# ==========================================================
# 5️⃣ Visualização opcional de deslocamentos
# ==========================================================
# Mostra a variação posicional média por resíduo (picos = regiões flexíveis)
fluctuations = calcSqFlucts(anm[:10])  # primeiros 10 modos
resnums = calphas.getResnums()

plt.figure(figsize=(8,4))
plt.plot(resnums, fluctuations, color='royalblue')
plt.xlabel("Residue index")
plt.ylabel("Mean-square fluctuation (Å²)")
plt.title("Residue flexibility from NMA (top 10 modes)")
plt.tight_layout()
plt.show()